# 🐺 WolfPicture Printify Studio 2026

Bu Google Colab defteri, AI ile üretilen tasarımları **baskıya hazırlamak, analiz etmek ve raporlamak** için hazırlanmıştır.

## İş akışı

1. Güvenli kurulum  
2. Oturumu doğrulama  
3. Ayarlar  
4. Görsel yükleme  
5. Ham görsel analizi  
6. Tişört rengi analizi  
7. Baskı boyu önerisi  
8. Arka plan silme  
9. Real-ESRGAN upscale  
10. Son kalite kontrolü  
11. Mockup oluşturma  
12. CSV + JSON raporu  
13. ZIP indirme  

> Kod hücreleri form görünümündedir. Yukarıdan aşağıya sırayla çalıştır.

## 1️⃣ Güvenli kurulum

Bu hücre, Pillow ve diğer paketleri uyumlu sürümlerle kurar.  
İlk çalıştırmadan sonra Colab oturumunu yeniden başlatman istenebilir.

In [ ]:
#@title 1️⃣ Güvenli Kurulum
import sys, subprocess, os, pkgutil, json, textwrap

PINNED = [
    "Pillow==11.3.0",
    "numpy==2.0.2",
    "pandas==2.2.2",
    "matplotlib==3.10.0",
    "opencv-python-headless==4.10.0.84",
    "rembg==2.0.67",
    "onnxruntime==1.22.1",
    "scipy==1.14.1"
]

cmd = [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall"] + PINNED
result = subprocess.run(cmd, text=True, capture_output=True)

if result.returncode != 0:
    print("❌ Kurulum hatası oluştu.")
    print(result.stderr[-4000:])
    raise RuntimeError("Paket kurulumu tamamlanamadı.")

print("✅ Temel paketler kuruldu.")
print("⚠️ Bu hücreyi ilk kez çalıştırdıysan:")
print("   Runtime > Restart session")
print("   ardından 2. adımdan devam et.")

## 2️⃣ Oturum doğrulama

Bu adım Pillow, OpenCV ve diğer temel paketlerin doğru yüklendiğini kontrol eder.

In [ ]:
#@title 2️⃣ Oturum Doğrulama
import sys, importlib, platform

checks = {}

try:
    import PIL
    from PIL import Image, ImageDraw, ImageFont, ImageOps, ImageEnhance, ImageFilter
    checks["Pillow"] = PIL.__version__
except Exception as e:
    checks["Pillow"] = f"HATA: {e}"

try:
    import cv2
    checks["OpenCV"] = cv2.__version__
except Exception as e:
    checks["OpenCV"] = f"HATA: {e}"

try:
    import numpy as np
    checks["NumPy"] = np.__version__
except Exception as e:
    checks["NumPy"] = f"HATA: {e}"

try:
    import pandas as pd
    checks["Pandas"] = pd.__version__
except Exception as e:
    checks["Pandas"] = f"HATA: {e}"

print("🐍 Python:", sys.version.split()[0])
print("💻 Sistem:", platform.platform())
for name, version in checks.items():
    print(f"{name}: {version}")

errors = [v for v in checks.values() if str(v).startswith("HATA:")]
if errors:
    print("\n❌ Bir paket doğru yüklenmemiş.")
    print("1. adımı tekrar çalıştır, ardından Runtime > Restart session yap.")
    raise RuntimeError("Ortam doğrulaması başarısız.")
else:
    print("\n✅ Ortam hazır.")

## 3️⃣ Proje ayarları

Burada yalnızca istediğin seçenekleri değiştir.

In [ ]:
#@title 3️⃣ Proje Ayarları
from pathlib import Path

target_dpi = 300 #@param {type:"integer"}
upscale_factor = 4 #@param [2, 4]
remove_background = True #@param {type:"boolean"}
create_mockups = True #@param {type:"boolean"}
max_file_size_mb = 100 #@param {type:"integer"}
default_recommended_width_cm = 30 #@param {type:"integer"}

WORKDIR = Path("/content/WolfPicture_Studio")
RAW_DIR = WORKDIR / "01_raw"
REMOVED_DIR = WORKDIR / "02_background_removed"
UPSCALED_DIR = WORKDIR / "03_upscaled"
MOCKUP_DIR = WORKDIR / "04_mockups"
REPORT_DIR = WORKDIR / "05_reports"
TEMP_DIR = WORKDIR / "temp"

for folder in [RAW_DIR, REMOVED_DIR, UPSCALED_DIR, MOCKUP_DIR, REPORT_DIR, TEMP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ Proje klasörleri hazır.")
print("📁", WORKDIR)
print("🖨️ Hedef DPI:", target_dpi)
print("🔍 Upscale:", f"{upscale_factor}×")
print("🫥 Arka plan silme:", "Açık" if remove_background else "Kapalı")

## 4️⃣ Görsel yükleme

PNG, JPG, JPEG ve WEBP desteklenir.

In [ ]:
#@title 4️⃣ Görselleri Yükle
from google.colab import files
from pathlib import Path

uploaded = files.upload()
supported = {".png", ".jpg", ".jpeg", ".webp"}
raw_files = []

for filename, data in uploaded.items():
    ext = Path(filename).suffix.lower()
    if ext not in supported:
        print(f"⚠️ Desteklenmeyen dosya atlandı: {filename}")
        continue

    destination = RAW_DIR / Path(filename).name
    destination.write_bytes(data)
    raw_files.append(destination)
    print(f"✅ Yüklendi: {destination.name}")

raw_files = sorted(raw_files)

if not raw_files:
    raise RuntimeError("Desteklenen görsel yüklenmedi.")

print(f"\n📦 Toplam {len(raw_files)} görsel hazır.")

## 5️⃣ Analiz motoru

Bu motor şunları ölçer:

- Keskinlik
- Koyu ve çok açık alanlar
- İnce detay yoğunluğu
- Şeffaflık ve kenar kalıntısı
- Dosya boyutu
- Minimum güvenli baskı genişliği
- Teknik maksimum baskı genişliği

In [ ]:
#@title 5️⃣ Analiz Motoru
import os, math, json
import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont, ImageOps, ImageEnhance, ImageFilter
from IPython.display import display
import matplotlib.pyplot as plt

SUPPORTED = {".png", ".jpg", ".jpeg", ".webp"}

def load_rgba(path):
    with Image.open(path) as image:
        return np.array(image.convert("RGBA"))

def analysis_resize(rgba, max_side=1600):
    h, w = rgba.shape[:2]
    scale = min(1.0, max_side / max(h, w))
    if scale >= 1.0:
        return rgba.copy()
    return cv2.resize(
        rgba,
        (max(1, int(w * scale)), max(1, int(h * scale))),
        interpolation=cv2.INTER_AREA
    )

def visible_mask(rgba):
    alpha = rgba[:, :, 3]
    return alpha > 20

def sharpness_score(rgba):
    mask = visible_mask(rgba)
    if not np.any(mask):
        return 0.0
    gray = cv2.cvtColor(rgba[:, :, :3], cv2.COLOR_RGB2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return float(np.var(lap[mask]))

def tone_metrics(rgba):
    mask = visible_mask(rgba)
    if not np.any(mask):
        return {"mean": 0.0, "dark": 100.0, "near_black": 100.0, "near_white": 0.0}
    gray = cv2.cvtColor(rgba[:, :, :3], cv2.COLOR_RGB2GRAY)[mask]
    return {
        "mean": float(gray.mean()),
        "dark": float((gray < 45).mean() * 100),
        "near_black": float((gray < 8).mean() * 100),
        "near_white": float((gray > 247).mean() * 100)
    }

def detail_density(rgba):
    mask = visible_mask(rgba)
    if not np.any(mask):
        return 0.0
    gray = cv2.cvtColor(rgba[:, :, :3], cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 70, 170)
    edges[~mask] = 0
    return float((edges > 0).sum() / max(1, mask.sum()) * 100)

def transparency_metrics(rgba):
    alpha = rgba[:, :, 3]
    h, w = alpha.shape
    border_width = max(2, int(min(h, w) * 0.01))
    border = np.concatenate([
        alpha[:border_width, :].ravel(),
        alpha[-border_width:, :].ravel(),
        alpha[:, :border_width].ravel(),
        alpha[:, -border_width:].ravel()
    ])
    return {
        "fully_transparent": float((alpha == 0).mean() * 100),
        "semi_transparent": float(((alpha > 0) & (alpha < 245)).mean() * 100),
        "visible_border": float((border > 10).mean() * 100)
    }

def recommended_print_sizes(width_px, detail, dark):
    technical_max_cm = width_px / target_dpi * 2.54

    if detail >= 26:
        minimum_cm = 30
    elif detail >= 22:
        minimum_cm = 28
    elif detail >= 17:
        minimum_cm = 24
    elif detail >= 10:
        minimum_cm = 21
    else:
        minimum_cm = 18

    if dark >= 45:
        minimum_cm += 2

    recommended_cm = max(minimum_cm + 2, default_recommended_width_cm)
    recommended_cm = min(recommended_cm, technical_max_cm)

    return round(minimum_cm, 1), round(recommended_cm, 1), round(technical_max_cm, 1)

def quality_decision(metrics):
    failures = []
    warnings = []

    if metrics["short_side"] < 2000:
        failures.append("Çözünürlük düşük")
    elif metrics["short_side"] < 3000:
        warnings.append("Çözünürlük orta seviyede")

    if metrics["sharpness"] < 30:
        failures.append("Belirgin bulanıklık")
    elif metrics["sharpness"] < 60:
        warnings.append("Keskinlik düşük olabilir")

    if metrics["dark"] > 60:
        warnings.append("Koyu alan oranı çok yüksek")
    elif metrics["dark"] > 38:
        warnings.append("Koyu alan oranı yüksek")

    if metrics["detail"] > 26:
        warnings.append("Çok ince detay yoğunluğu")
    elif metrics["detail"] > 21:
        warnings.append("İnce detay yoğunluğu yüksek")

    if metrics["file_mb"] > max_file_size_mb:
        failures.append("Dosya boyutu sınırı aşıyor")

    if len(failures) >= 2:
        return "FAIL", "❌ YENİDEN İŞLE", 45, failures, warnings
    if len(failures) == 1:
        return "CHECK", "⚠️ KONTROL ET", 68, failures, warnings
    if len(warnings) >= 3:
        return "CHECK", "⚠️ KONTROL ET", 74, failures, warnings
    if len(warnings) == 2:
        return "READY_CHECK", "✅ BASKIYA UYGUN — KÜÇÜK KONTROL", 86, failures, warnings
    if len(warnings) == 1:
        return "READY_CHECK", "✅ BASKIYA UYGUN — KÜÇÜK KONTROL", 94, failures, warnings
    return "READY", "✅ BASKIYA HAZIR", 100, failures, warnings

def analyze_image(path):
    full_rgba = load_rgba(path)
    rgba = analysis_resize(full_rgba)
    h, w = full_rgba.shape[:2]

    tones = tone_metrics(rgba)
    transparency = transparency_metrics(rgba)
    details = detail_density(rgba)
    sharpness = sharpness_score(rgba)
    file_mb = os.path.getsize(path) / (1024 ** 2)
    min_cm, recommended_cm, max_cm = recommended_print_sizes(w, details, tones["dark"])

    metrics = {
        "file": str(path),
        "width": int(w),
        "height": int(h),
        "short_side": int(min(w, h)),
        "file_mb": float(file_mb),
        "sharpness": float(sharpness),
        "mean_brightness": tones["mean"],
        "dark": tones["dark"],
        "near_black": tones["near_black"],
        "near_white": tones["near_white"],
        "detail": float(details),
        "transparent": transparency["fully_transparent"],
        "semi_transparent": transparency["semi_transparent"],
        "visible_border": transparency["visible_border"],
        "minimum_print_cm": min_cm,
        "recommended_print_cm": recommended_cm,
        "technical_max_cm": max_cm
    }

    code, decision, score, failures, warnings = quality_decision(metrics)
    metrics.update({
        "code": code,
        "decision": decision,
        "score": score,
        "failures": failures,
        "warnings": warnings
    })
    return metrics

print("✅ Analiz motoru hazır.")

## 6️⃣ Ham görsel analizi

Bu adım, işlem yapmadan önce sorunları yakalar.

In [ ]:
#@title 6️⃣ Ham Görsel Analizi
raw_results = []

for path in raw_files:
    result = analyze_image(path)
    raw_results.append(result)

    print("\n" + "=" * 78)
    print(f"🖼️ {path.name}")
    print(f"🎯 {result['decision']} — {result['score']}/100")
    print(f"📐 {result['width']} × {result['height']} px")
    print(f"🔍 Keskinlik: {result['sharpness']:.1f}")
    print(f"🌑 Koyu alan: %{result['dark']:.1f}")
    print(f"🧵 Detay yoğunluğu: %{result['detail']:.1f}")
    print(f"📏 Minimum güvenli baskı: {result['minimum_print_cm']} cm")
    print(f"✨ Önerilen baskı: {result['recommended_print_cm']} cm")
    print(f"🖨️ Teknik maksimum: {result['technical_max_cm']} cm")

    for warning in result["warnings"]:
        print(f"⚠️ {warning}")
    for failure in result["failures"]:
        print(f"❌ {failure}")

## 7️⃣ Tişört renk analizi

Görseli farklı tişört renkleri üzerinde simüle eder ve kontrast puanı verir.

In [ ]:
#@title 7️⃣ Tişört Rengi Analizi
GARMENT_COLORS = {
    "Beyaz": (255, 255, 255),
    "Krem": (243, 234, 214),
    "Doğal": (225, 210, 180),
    "Kum": (204, 184, 146),
    "Açık Gri": (205, 205, 205),
    "Koyu Gri": (70, 70, 70),
    "Siyah": (18, 18, 18),
    "Lacivert": (28, 38, 72),
    "Bordo": (100, 30, 45),
    "Orman Yeşili": (48, 75, 52),
    "Kahverengi": (90, 60, 42)
}

def composite_on_background(rgba, bg_rgb):
    rgb = rgba[:, :, :3].astype(np.float32)
    alpha = rgba[:, :, 3].astype(np.float32) / 255.0
    bg = np.empty_like(rgb)
    bg[:] = bg_rgb
    return rgb * alpha[..., None] + bg * (1.0 - alpha[..., None])

def contrast_score(rgba, bg_rgb):
    mask = visible_mask(rgba)
    if not np.any(mask):
        return 0.0

    composite = composite_on_background(rgba, bg_rgb).astype(np.uint8)
    gray = cv2.cvtColor(composite, cv2.COLOR_RGB2GRAY)
    bg_gray = 0.299 * bg_rgb[0] + 0.587 * bg_rgb[1] + 0.114 * bg_rgb[2]
    difference = np.abs(gray.astype(np.float32) - bg_gray)

    mean_difference = float(np.mean(difference[mask]))
    low_contrast_ratio = float((difference[mask] < 25).mean() * 100)

    score = (mean_difference / 1.2) - (low_contrast_ratio * 0.35)
    return round(max(0.0, min(100.0, score)), 1)

color_results = {}

for path in raw_files:
    rgba = analysis_resize(load_rgba(path))
    scores = {
        name: contrast_score(rgba, rgb)
        for name, rgb in GARMENT_COLORS.items()
    }
    ranking = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    color_results[path.name] = ranking

    print("\n" + "=" * 78)
    print(f"👕 {path.name}")
    print("🏆 Önerilen renkler:")
    for name, score in ranking[:4]:
        print(f"   ✅ {name}: {score}/100")
    print("🚫 Riskli renkler:")
    for name, score in ranking[-3:]:
        print(f"   ⚠️ {name}: {score}/100")

## 8️⃣ Baskı boyu önerisi

Bu sonuç evrensel bir baskı kuralı değildir.  
Görselin detay yoğunluğu ve çözünürlüğüne göre güvenli bir tahmindir.

In [ ]:
#@title 8️⃣ Baskı Boyu Önerisi
for result in raw_results:
    print("\n" + "=" * 78)
    print(f"📄 {Path(result['file']).name}")
    print(f"📏 Minimum güvenli genişlik: {result['minimum_print_cm']} cm")
    print(f"✨ Önerilen genişlik: {result['recommended_print_cm']} cm")
    print(f"🖨️ 300 DPI teknik maksimum: {result['technical_max_cm']} cm")

    if result["detail"] >= 26:
        print("🚫 Küçük sol göğüs baskısı önerilmez.")
        print("✅ Büyük ön baskı önerilir.")
    elif result["detail"] >= 18:
        print("ℹ️ Orta veya büyük baskı kullan.")
    else:
        print("✅ Küçük veya orta baskıya daha uygundur.")

## 9️⃣ Arka plan silme

`rembg` ile şeffaf PNG üretir.

In [ ]:
#@title 9️⃣ Arka Planı Sil
from rembg import remove

removed_files = []

for path in raw_files:
    output_path = REMOVED_DIR / f"{path.stem}_rembg.png"

    try:
        if remove_background:
            output_path.write_bytes(remove(path.read_bytes()))
            print(f"✅ Arka plan silindi: {output_path.name}")
        else:
            with Image.open(path) as image:
                image.convert("RGBA").save(output_path)
            print(f"ℹ️ Arka plan silme kapalı: {output_path.name}")

        removed_files.append(output_path)
    except Exception as error:
        print(f"❌ {path.name}: {error}")

if not removed_files:
    raise RuntimeError("Arka plan silme aşamasında çıktı üretilemedi.")

print(f"\n📦 {len(removed_files)} şeffaf PNG hazır.")

## 🔟 Real-ESRGAN kurulumu ve upscale

Bu adım resmi 2× veya 4× büyütür.  
Face Restore kullanılmaz.

In [ ]:
#@title 🔟 Real-ESRGAN Kurulumu
import subprocess, sys, urllib.request, shutil

REALESRGAN_DIR = Path("/content/Real-ESRGAN")
WEIGHTS_DIR = REALESRGAN_DIR / "weights"
WEIGHT_FILE = WEIGHTS_DIR / "RealESRGAN_x4plus.pth"

if not REALESRGAN_DIR.exists():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/xinntao/Real-ESRGAN.git", str(REALESRGAN_DIR)],
        check=True
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "basicsr==1.4.2", "facexlib==0.3.0", "gfpgan==1.3.8"],
    check=True
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REALESRGAN_DIR / "requirements.txt")],
    check=True
)

subprocess.run(
    [sys.executable, "setup.py", "develop", "-q"],
    cwd=REALESRGAN_DIR,
    check=True
)

WEIGHTS_DIR.mkdir(exist_ok=True)

if not WEIGHT_FILE.exists():
    urllib.request.urlretrieve(
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/RealESRGAN_x4plus.pth",
        WEIGHT_FILE
    )

print("✅ Real-ESRGAN hazır.")

In [ ]:
#@title 🔟 Upscale İşlemini Başlat
upscaled_files = []

for path in removed_files:
    command = [
        sys.executable,
        str(REALESRGAN_DIR / "inference_realesrgan.py"),
        "-n", "RealESRGAN_x4plus",
        "-i", str(path),
        "-o", str(UPSCALED_DIR),
        "-s", str(upscale_factor),
        "--ext", "png"
    ]

    try:
        subprocess.run(command, cwd=REALESRGAN_DIR, check=True)

        expected = UPSCALED_DIR / f"{path.stem}_out.png"
        final_path = UPSCALED_DIR / f"{path.stem}_upscaled.png"

        if expected.exists():
            expected.replace(final_path)
        else:
            candidates = sorted(UPSCALED_DIR.glob(f"{path.stem}*.png"))
            if not candidates:
                raise FileNotFoundError("Upscale çıktısı bulunamadı.")
            candidates[0].replace(final_path)

        with Image.open(final_path) as image:
            image.convert("RGBA").save(final_path, dpi=(target_dpi, target_dpi))

        upscaled_files.append(final_path)
        print(f"✅ {final_path.name}")

    except Exception as error:
        print(f"❌ {path.name}: {error}")

if not upscaled_files:
    raise RuntimeError("Upscale çıktısı üretilemedi.")

print(f"\n📦 {len(upscaled_files)} upscale PNG hazır.")

## 1️⃣1️⃣ Son kalite kontrolü

Upscale ve arka plan silme sonrasında dosyaları tekrar analiz eder.

In [ ]:
#@title 1️⃣1️⃣ Son Kalite Kontrolü
final_results = []

for path in upscaled_files:
    result = analyze_image(path)
    final_results.append(result)

    print("\n" + "=" * 78)
    print(f"🐺 {path.name}")
    print(f"🎯 {result['decision']} — {result['score']}/100")
    print(f"📐 {result['width']} × {result['height']} px")
    print(f"📏 Minimum baskı: {result['minimum_print_cm']} cm")
    print(f"✨ Önerilen baskı: {result['recommended_print_cm']} cm")
    print(f"🖨️ Teknik maksimum: {result['technical_max_cm']} cm")
    print(f"🌑 Koyu alan: %{result['dark']:.1f}")
    print(f"🧵 Detay yoğunluğu: %{result['detail']:.1f}")
    print(f"🫥 Dış kenar görünür piksel: %{result['visible_border']:.2f}")
    print(f"📦 Dosya: {result['file_mb']:.2f} MB")

    for warning in result["warnings"]:
        print(f"⚠️ {warning}")
    for failure in result["failures"]:
        print(f"❌ {failure}")

## 1️⃣2️⃣ Mockup oluşturma

Bu adım düz renk tişört ön izlemeleri üretir.  
Nihai mağaza mockup'ı için Printify Product Creator kullanılmalıdır.

In [ ]:
#@title 1️⃣2️⃣ Mockup Oluştur
def make_mockup(design_path, garment_rgb, garment_name, output_path):
    canvas_w, canvas_h = 1600, 1800
    canvas = Image.new("RGB", (canvas_w, canvas_h), garment_rgb)
    draw = ImageDraw.Draw(canvas)

    # Basit tişört silueti
    outline = tuple(max(0, min(255, c - 35 if sum(garment_rgb) > 380 else c + 50)) for c in garment_rgb)
    shirt = [
        (420, 180), (610, 95), (760, 170), (840, 170), (990, 95), (1180, 180),
        (1450, 520), (1220, 680), (1160, 520), (1160, 1600),
        (440, 1600), (440, 520), (380, 680), (150, 520)
    ]
    draw.polygon(shirt, fill=garment_rgb, outline=outline)

    with Image.open(design_path) as image:
        design = image.convert("RGBA")
        design.thumbnail((720, 920), Image.Resampling.LANCZOS)

    x = (canvas_w - design.width) // 2
    y = 380
    canvas.paste(design, (x, y), design)

    draw.rounded_rectangle(
        (60, 60, canvas_w - 60, canvas_h - 60),
        radius=30,
        outline=(130, 130, 130),
        width=4
    )
    draw.text((90, canvas_h - 115), f"{garment_name} tişört ön izlemesi", fill=outline)

    canvas.save(output_path, quality=95)

mockup_files = []

if create_mockups:
    for final_path in upscaled_files:
        base_name = final_path.name.replace("_rembg_upscaled.png", "")
        raw_key = next(
            (name for name in color_results if Path(name).stem == base_name or Path(name).stem in final_path.stem),
            None
        )

        ranking = color_results.get(raw_key, [])
        recommended_names = [name for name, _ in ranking[:4]] or ["Beyaz", "Krem", "Açık Gri", "Siyah"]

        for garment_name in recommended_names:
            output_path = MOCKUP_DIR / f"{final_path.stem}_{garment_name.replace(' ', '_')}.jpg"
            make_mockup(
                final_path,
                GARMENT_COLORS[garment_name],
                garment_name,
                output_path
            )
            mockup_files.append(output_path)

    print(f"✅ {len(mockup_files)} mockup oluşturuldu.")

    for mockup in mockup_files[:8]:
        display(Image.open(mockup).resize((280, 315)))
else:
    print("ℹ️ Mockup oluşturma kapalı.")

## 1️⃣3️⃣ CSV, JSON ve ZIP

Bütün sonuçları raporlar ve tek ZIP dosyasında indirir.

In [ ]:
#@title 1️⃣3️⃣ Raporları Oluştur
rows = []

for result in final_results:
    filename = Path(result["file"]).name
    raw_key = next(
        (name for name in color_results if Path(name).stem in filename),
        None
    )
    best_colors = ", ".join(name for name, _ in color_results.get(raw_key, [])[:4])

    rows.append({
        "Dosya": filename,
        "Karar": result["decision"],
        "Puan": result["score"],
        "Boyut": f"{result['width']}×{result['height']}",
        "Minimum Baskı cm": result["minimum_print_cm"],
        "Önerilen Baskı cm": result["recommended_print_cm"],
        "Teknik Maksimum cm": result["technical_max_cm"],
        "En İyi Tişört Renkleri": best_colors,
        "Koyu Alan %": round(result["dark"], 1),
        "Detay Yoğunluğu %": round(result["detail"], 1),
        "Dosya MB": round(result["file_mb"], 2)
    })

report_df = pd.DataFrame(rows)
display(report_df)

csv_path = REPORT_DIR / "WolfPicture_Printify_Report.csv"
json_path = REPORT_DIR / "WolfPicture_Printify_Report.json"

report_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
json_path.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")

print("✅ CSV:", csv_path)
print("✅ JSON:", json_path)

In [ ]:
#@title 1️⃣3️⃣ ZIP Paketini İndir
from google.colab import files
import zipfile

zip_path = Path("/content/WolfPicture_Printify_Ready.zip")

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in [UPSCALED_DIR, MOCKUP_DIR, REPORT_DIR]:
        for file in folder.rglob("*"):
            if file.is_file():
                archive.write(file, file.relative_to(WORKDIR))

print("✅ ZIP hazır:", zip_path)
files.download(str(zip_path))

# ✅ İşlem tamamlandı

Üretilen dosyalar:

- Şeffaf PNG
- Real-ESRGAN upscale PNG
- Tişört rengi önerileri
- Baskı boyu önerileri
- Son kalite raporu
- Mockuplar
- CSV
- JSON
- ZIP

> Printify'a yüklemeden önce seçtiğin ürünün gerçek baskı alanını Product Creator içinde son kez kontrol et.